# Inductive biases for transitive inference
Now that we have a geometric solution for transitive inference in `geometric_solution.ipynb`, how can we bias our networks to learn this solution? In this notebook, we'll bias them to learn stimulus embeddings that lie on the unit circle. In effect, if the only parameter the network can tune is the angle of the embedding, then perhaps the network is biased towards learning a solution that generalizes.

## Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from relational_attention.tasks import TransitiveInferenceDataset

In [2]:
torch.manual_seed(42)

## Build task

In [3]:
# Parameters
num_samples = 1000
num_items = 5

In [4]:
# Training task
dataset_train = TransitiveInferenceDataset(num_samples=num_samples, num_items=num_items)
loader_train = DataLoader(dataset_train, batch_size=16, shuffle=True)

In [5]:
# Testing task
dataset_test = TransitiveInferenceDataset(num_samples=num_samples, num_items=num_items, train=False)
loader_test = DataLoader(dataset_test, batch_size=16, shuffle=True)

## Build models

### Minimal model

In [6]:
class RelationalAttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim):
        super().__init__()
        # Initialize weights
        self.W_q = nn.Linear(embed_dim, head_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, head_dim, bias=False)

    def forward(self, x_1, x_2):
        """
        x_1: (batch, embed_dim)
        x_2: (batch, embed_dim)
        returns: (batch, seq_len, head_dim)
        """
        Q = self.W_q(x_2)                            # (B, D)
        K = self.W_k(x_1)                            # (B, D)

        # Compute attention scores over the *learned* Value slots
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) / (K.size(-1) ** 0.5)  # (B, L, L)
        return attn_weights.flatten()

### Normalized model

In [7]:
class UnitNormLinear(nn.Module):
    def __init__(self, in_features, out_features, dim_to_normalize=0, bias=False, eps=1e-8):
        super().__init__()
        self.weight_raw = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        self.dim = dim_to_normalize  # 0 = rows, 1 = columns
        self.eps = eps

    def forward(self, x):
        # weight_raw shape: (out_features, in_features)
        # Normalize along chosen dim
        w = self.weight_raw
        norm = w.norm(dim=self.dim, keepdim=True).clamp_min(self.eps)
        w_hat = w / norm

        # Standard linear: x @ W^T
        out = x @ w_hat.t()
        if self.bias is not None:
            out = out + self.bias
        return out

In [8]:
class RelationalNormAttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim):
        super().__init__()
        # Initialize weights
        self.W_q = UnitNormLinear(embed_dim, head_dim, dim_to_normalize=1, bias=False)
        self.W_k = UnitNormLinear(embed_dim, head_dim, dim_to_normalize=1, bias=False)

    def forward(self, x_1, x_2):
        """
        x_1: (batch, embed_dim)
        x_2: (batch, embed_dim)
        returns: (batch, seq_len, head_dim)
        """
        Q = self.W_q(x_2)                            # (B, D)
        K = self.W_k(x_1)                            # (B, D)

        # Compute attention scores over the *learned* Value slots
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) / (K.size(-1) ** 0.5)  # (B, L, L)
        return attn_weights.flatten()

### Norm penalty model

In [9]:
def unit_norm_penalty(W, dim=1):
    # W shape: (out, in) – encourage columns to be unit norm if dim=0, rows if dim=1
    norms = W.norm(dim=dim)
    return ((norms - 1.0) ** 2).mean()